# dna-interp: what did the model learn?

This notebook combines the saved analysis arrays into one narrative. Run
`scripts/run_synthetic_demo.py` (offline) or the real DNABERT-2 pipeline first;
both write `.npz` arrays to `results/figures/` and a summary JSON to
`results/cache/`. The cells below reload those artifacts so figures regenerate
without re-running the experiments.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib.pyplot as plt
FIG = Path.cwd().parent / 'results' / 'figures'
summary = json.load(open(Path.cwd().parent / 'results' / 'cache' / 'synthetic_demo_summary.json'))
print('test metrics:', summary['test_metrics'], '\nnote:', summary['note'])

## Stage 2: attention head specialization

In [ ]:
a = np.load(FIG / 'attention_heads.npz')
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = ax[0].imshow(a['mean_entropy'], aspect='auto', cmap='viridis_r', origin='lower'); ax[0].set_title('mean entropy (low=specialized)'); ax[0].set_xlabel('head'); ax[0].set_ylabel('layer'); fig.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(a['task_correlation'], aspect='auto', cmap='coolwarm', origin='lower'); ax[1].set_title('entropy vs confidence (Spearman)'); ax[1].set_xlabel('head'); ax[1].set_ylabel('layer'); fig.colorbar(im1, ax=ax[1])
plt.tight_layout(); plt.show()

## Stage 3: activation patching (causal localization)

In [ ]:
p = np.load(FIG / 'patching.npz')
h = p['heatmap']; m = np.abs(h).max()
plt.figure(figsize=(10, 4)); plt.imshow(h, aspect='auto', cmap='coolwarm', origin='lower', vmin=-m, vmax=m)
plt.yticks(range(len(p['layers'])), p['layers']); plt.xlabel('token position'); plt.ylabel('layer'); plt.title('Delta enhancer logit'); plt.colorbar(); plt.show()
print('top causal sites:', summary['top_causal_sites'])
print('causal near planted motif:', summary['causal_near_motif'])

## Stage 4: layer-wise probing

In [ ]:
pr = np.load(FIG / 'probing.npz', allow_pickle=True)
mat, names = pr['matrix'], list(pr['properties'])
plt.figure(figsize=(9, 0.7*len(names)+1.5)); plt.imshow(mat, aspect='auto', cmap='magma', vmin=0, vmax=1)
plt.yticks(range(len(names)), names); plt.xlabel('layer (0=embedding)'); plt.title('probe metric by layer')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]): plt.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', color='w', fontsize=7)
plt.colorbar(); plt.show()

## Stage 5: JASPAR motif vs attention

In [ ]:
for head, top in summary['motif_correlations'].items():
    print(head, '->', [(n, round(c, 3)) for n, c in top])
al = np.load(FIG / 'motif_alignment.npz')
att, trk = al['attention'], al['motif_track']
plt.figure(figsize=(10, 3)); plt.plot(att/(att.max()+1e-9), label='head attention'); plt.plot(trk/(trk.max()+1e-9), label='motif log-odds'); plt.legend(); plt.xlabel('token'); plt.title('head attention vs motif occurrence'); plt.show()

## Conclusions

Fill in from the summary above. For the synthetic validation the pattern is:
the model solves the task via surface statistics (GC content is linearly
decodable at every layer, including the embedding), causal effect concentrates
in early layers at motif-bearing positions, attention heads stay high-entropy
(not sharply specialized), and the heads correlate with the GC-rich SP1 motif
while staying uncorrelated with the decoy control. That is the "learns
statistical patterns, not deep TF grammar" outcome, which the same code will
test on DNABERT-2 + real BEND data once the assets are downloaded.